# Humvee component detector: initialization comparison

## Goal

Train the same YOLOX-S detector twice on the supplied Humvee component export:

1. from random weights;
2. from the promoted Carparts checkpoint, carparts23-yolox-s-coco-v1.

The data split, export classes, seed, augmentation, schedule, evaluator, and hardware settings are shared. Only initialization changes. Comparison uses validation COCO AP; the test split remains reserved for one final evaluation after an approach is selected.

**Execution status:** intentionally unrun. No cells or training jobs were executed when this notebook was created.


## Setup

Run this notebook with the repository's Python 3.10 training environment. It imports the pinned YOLOX checkout directly, so an editable YOLOX installation is not required, but the environment still needs YOLOX dependencies, PyTorch with CUDA, pycocotools, TensorBoard, and Jupyter.

The notebook discovers the repository root, validates the source COCO export, writes deterministic split annotations and manifests, checks checkpoint compatibility, and then exposes one guarded cell that launches both runs. The current repository environment does not include Jupyter; install and launch it without executing the notebook automatically:

    .\.venv\Scripts\python.exe -m pip install jupyterlab ipykernel
    .\.venv\Scripts\python.exe -m jupyter lab


In [ ]:
from __future__ import annotations

import argparse
import csv
import gc
import hashlib
import json
import random
import sys
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch


def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "third_party" / "YOLOX" / "yolox").is_dir()
            and (candidate / "configs" / "taxonomy.json").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find the CRATER repository root from the current directory."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
DETECTION_DIR = REPOSITORY_ROOT / "experiments" / "detection"
YOLOX_ROOT = REPOSITORY_ROOT / "third_party" / "YOLOX"

for import_root in (YOLOX_ROOT, DETECTION_DIR):
    import_path = str(import_root)
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

from yolox.core import Trainer
from yolox.data import COCODataset, TrainTransform, ValTransform
from yolox.exp import Exp as YOLOXExp
from yolox.utils import configure_module
from standard_coco_evaluator import StandardCOCOEvaluator

configure_module()
print(f"Repository: {REPOSITORY_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


### Parameters

The category names are loaded from instances_default.json and are never mapped to the canonical CRATER taxonomy.

GROUP_COLUMN is None because the current source manifest has no populated provenance groups. That produces a deterministic image-level split suitable for an exploratory initialization comparison, but not for release-quality performance claims. Once source grouping is populated, set GROUP_COLUMN to source_asset_id, scene_id, vehicle_id, or sequence_id and regenerate the splits.


In [ ]:
SEED = 42

SOURCE_DIRECTORY = (
    REPOSITORY_ROOT
    / "datasets"
    / "military"
    / "components"
    / "source"
    / "humvee"
)
SOURCE_ANNOTATIONS = SOURCE_DIRECTORY / "instances_default.json"
SOURCE_MANIFEST = SOURCE_DIRECTORY / "source_manifest.csv"

DATASET_ROOT = REPOSITORY_ROOT / "datasets" / "military" / "components"
ANNOTATION_DIRECTORY = DATASET_ROOT / "annotations"
SPLIT_MANIFEST_DIRECTORY = DATASET_ROOT / "manifests"

ANNOTATION_FILES = {
    "train": "humvee_source6_instances_train.json",
    "val": "humvee_source6_instances_val.json",
    "test": "humvee_source6_instances_test.json",
}
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
GROUP_COLUMN: str | None = None

OUTPUT_ROOT = REPOSITORY_ROOT / "outputs" / "detection" / "humvee_source6"

INPUT_SIZE = (640, 640)
TEST_SIZE = (640, 640)
MAX_EPOCHS = 100
WARMUP_EPOCHS = 5
NO_AUG_EPOCHS = 15
EVAL_INTERVAL = 5
BATCH_SIZE = 8
DATA_WORKERS = 4
FP16 = True

CARPARTS_PARENT_VERSION = "carparts23-yolox-s-coco-v1"
CARPARTS_CHECKPOINT = (
    REPOSITORY_ROOT
    / "outputs"
    / "detection"
    / "civilian"
    / "promoted"
    / CARPARTS_PARENT_VERSION
    / f"{CARPARTS_PARENT_VERSION}.pth"
)
EXPECTED_CARPARTS_CHECKPOINT_SHA256 = (
    "5AB8C99A5A67CA253AEA6F1D4A6974F827B5E5BD16ACA227306C129A7FD94D47"
)

# Safety switch. Leave False while reviewing or preparing data.
RUN_TRAINING = False

assert abs(sum(SPLIT_FRACTIONS.values()) - 1.0) < 1e-12


## Steps

### 1. Validate and split the source export

The split builder keeps the received category IDs and names unchanged. Generated COCO files point back to the ignored source images, so the notebook does not duplicate roughly 105 MiB of imagery.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def load_source() -> tuple[dict[str, Any], dict[str, dict[str, str]]]:
    payload = json.loads(SOURCE_ANNOTATIONS.read_text(encoding="utf-8"))
    with SOURCE_MANIFEST.open("r", encoding="utf-8-sig", newline="") as handle:
        manifest = {row["file_name"]: row for row in csv.DictReader(handle)}

    required_keys = {"images", "annotations", "categories"}
    missing_keys = required_keys.difference(payload)
    if missing_keys:
        raise ValueError(f"COCO export is missing keys: {sorted(missing_keys)}")

    category_ids = [int(category["id"]) for category in payload["categories"]]
    if len(category_ids) != len(set(category_ids)):
        raise ValueError("COCO category IDs are not unique.")

    image_ids = {int(image["id"]) for image in payload["images"]}
    image_by_id = {
        int(image["id"]): image for image in payload["images"]
    }
    image_names = {image["file_name"] for image in payload["images"]}
    if image_names != set(manifest):
        missing = sorted(image_names.difference(manifest))
        extra = sorted(set(manifest).difference(image_names))
        raise ValueError(
            f"Source manifest mismatch. Missing={missing[:5]}, extra={extra[:5]}"
        )

    for image in payload["images"]:
        image_path = SOURCE_DIRECTORY / image["file_name"]
        if not image_path.is_file():
            raise FileNotFoundError(image_path)

    for annotation in payload["annotations"]:
        if int(annotation["image_id"]) not in image_ids:
            raise ValueError(f"Unknown image_id in annotation {annotation['id']}")
        if int(annotation["category_id"]) not in category_ids:
            raise ValueError(f"Unknown category_id in annotation {annotation['id']}")
        if len(annotation["bbox"]) != 4:
            raise ValueError(f"Invalid bbox in annotation {annotation['id']}")
        x, y, width, height = map(float, annotation["bbox"])
        image = image_by_id[int(annotation["image_id"])]
        outside_image = (
            x < 0
            or y < 0
            or width <= 0
            or height <= 0
            or x + width > float(image["width"]) + 1e-6
            or y + height > float(image["height"]) + 1e-6
        )
        if outside_image:
            raise ValueError(f"Invalid bbox in annotation {annotation['id']}")

    return payload, manifest


#### Build deterministic splits

GROUP_COLUMN controls whether images are kept together by a provenance group. The same generated split files are shared by both training runs.


In [ ]:
def assign_groups(
    payload: dict[str, Any],
    manifest: dict[str, dict[str, str]],
) -> dict[int, str]:
    groups: dict[str, list[int]] = defaultdict(list)

    for image in payload["images"]:
        file_name = image["file_name"]
        if GROUP_COLUMN is None:
            group_id = file_name
        else:
            if GROUP_COLUMN not in manifest[file_name]:
                raise KeyError(f"Manifest has no column named {GROUP_COLUMN!r}")
            group_id = manifest[file_name][GROUP_COLUMN].strip()
            if not group_id:
                raise ValueError(
                    f"{GROUP_COLUMN} is blank for {file_name}. "
                    "Populate grouping metadata or set GROUP_COLUMN=None "
                    "for an exploratory image-level split."
                )
        groups[group_id].append(int(image["id"]))

    if len(groups) < len(SPLIT_FRACTIONS):
        raise ValueError("At least three independent groups are required.")

    rng = random.Random(SEED)
    ordered_groups = list(groups.items())
    rng.shuffle(ordered_groups)
    ordered_groups.sort(key=lambda item: len(item[1]), reverse=True)

    image_target = {
        split: len(payload["images"]) * fraction
        for split, fraction in SPLIT_FRACTIONS.items()
    }
    image_count = {split: 0 for split in SPLIT_FRACTIONS}
    assignment: dict[int, str] = {}

    split_names = list(SPLIT_FRACTIONS)
    for index, (_, image_ids) in enumerate(ordered_groups):
        if index < len(split_names):
            split = split_names[index]
        else:
            split = max(
                split_names,
                key=lambda name: (
                    image_target[name] - image_count[name]
                ) / max(image_target[name], 1),
            )
        for image_id in image_ids:
            assignment[image_id] = split
        image_count[split] += len(image_ids)

    return assignment


def write_splits(
    payload: dict[str, Any],
    assignment: dict[int, str],
) -> dict[str, dict[str, Any]]:
    ANNOTATION_DIRECTORY.mkdir(parents=True, exist_ok=True)
    SPLIT_MANIFEST_DIRECTORY.mkdir(parents=True, exist_ok=True)

    summaries: dict[str, dict[str, Any]] = {}
    source_prefix = SOURCE_DIRECTORY.relative_to(DATASET_ROOT).as_posix()

    for split, annotation_name in ANNOTATION_FILES.items():
        images = []
        split_image_ids = {
            image_id for image_id, assigned_split in assignment.items()
            if assigned_split == split
        }

        for source_image in payload["images"]:
            if int(source_image["id"]) not in split_image_ids:
                continue
            image = dict(source_image)
            image["file_name"] = f"{source_prefix}/{source_image['file_name']}"
            images.append(image)

        annotations = [
            annotation
            for annotation in payload["annotations"]
            if int(annotation["image_id"]) in split_image_ids
        ]

        output_payload = {
            "licenses": payload.get("licenses", []),
            "info": {
                **payload.get("info", {}),
                "description": (
                    "CRATER Humvee initialization comparison; "
                    f"{split} split; seed={SEED}; group_column={GROUP_COLUMN}"
                ),
                "source_annotation_sha256": sha256_file(SOURCE_ANNOTATIONS),
            },
            "categories": payload["categories"],
            "images": sorted(images, key=lambda item: int(item["id"])),
            "annotations": sorted(
                annotations, key=lambda item: int(item["id"])
            ),
        }

        annotation_path = ANNOTATION_DIRECTORY / annotation_name
        annotation_path.write_text(
            json.dumps(output_payload, indent=2) + "\n",
            encoding="utf-8",
            newline="\n",
        )

        file_names = [image["file_name"] for image in output_payload["images"]]
        manifest_path = (
            SPLIT_MANIFEST_DIRECTORY / f"humvee_source6_{split}.txt"
        )
        manifest_path.write_text(
            "\n".join(file_names) + "\n",
            encoding="utf-8",
            newline="\n",
        )

        category_counts = Counter(
            int(annotation["category_id"]) for annotation in annotations
        )
        summaries[split] = {
            "images": len(images),
            "annotations": len(annotations),
            "category_counts": dict(sorted(category_counts.items())),
            "annotation_path": str(annotation_path),
            "manifest_path": str(manifest_path),
        }

    return summaries


#### Materialize the shared data contract


In [ ]:
source_payload, source_manifest = load_source()
export_categories = tuple(
    category["name"]
    for category in sorted(
        source_payload["categories"], key=lambda item: int(item["id"])
    )
)
split_assignment = assign_groups(source_payload, source_manifest)
split_summaries = write_splits(source_payload, split_assignment)

print("Export classes:", export_categories)
print("Source annotation SHA-256:", sha256_file(SOURCE_ANNOTATIONS))
for split_name, summary in split_summaries.items():
    print(split_name, summary)


### 2. Verify the promoted Carparts checkpoint

Stage 1 selected carparts23-yolox-s-coco-v1 after the COCO-initialized run improved validation AP50:95 from 0.422 to 0.546 under the same 100-epoch schedule. The tracked Stage 1 report records its expected local path and SHA-256.

The checkpoint itself is intentionally ignored by Git, so merging main brings its results and lineage but not the binary. Restore it at the configured path before training. The preflight rejects a missing file, a wrong hash, a non-YOLOX payload, or a checkpoint with no compatible tensors. The 23-class prediction tensors are expected to be skipped when the six-class Humvee head is initialized.


In [ ]:
def discover_carparts_checkpoints() -> list[Path]:
    patterns = (
        "outputs/detection/civilian/promoted/**/*.pth",
        "outputs/detection/civilian/**/best_ckpt.pth",
        "outputs/yolox/yolox_s_carparts23*/best_ckpt.pth",
        "third_party/YOLOX/YOLOX_outputs/yolox_s_carparts23*/best_ckpt.pth",
    )
    candidates: set[Path] = set()
    for pattern in patterns:
        candidates.update(REPOSITORY_ROOT.glob(pattern))
    return sorted(path.resolve() for path in candidates if path.is_file())


checkpoint_candidates = discover_carparts_checkpoints()
print("Required promoted checkpoint:", CARPARTS_CHECKPOINT)
print("Expected SHA-256:", EXPECTED_CARPARTS_CHECKPOINT_SHA256)
if checkpoint_candidates:
    print("Discovered Carparts checkpoint candidates:")
    for checkpoint_candidate in checkpoint_candidates:
        print(" -", checkpoint_candidate)
else:
    print(
        "No local Carparts checkpoint found. Restore the promoted artifact "
        "before enabling training."
    )


### 3. Define the shared YOLOX experiment

This experiment reads the generated split annotations and uses standard pycocotools COCO evaluation. Both runs instantiate a fresh copy of this exact class.


In [ ]:
class HumveeSource6Experiment(YOLOXExp):
    def __init__(self):
        super().__init__()

        self.depth = 0.33
        self.width = 0.50
        self.num_classes = len(export_categories)

        self.data_dir = str(DATASET_ROOT)
        self.train_ann = ANNOTATION_FILES["train"]
        self.val_ann = ANNOTATION_FILES["val"]
        self.test_ann = ANNOTATION_FILES["test"]

        self.input_size = INPUT_SIZE
        self.test_size = TEST_SIZE
        self.max_epoch = MAX_EPOCHS
        self.warmup_epochs = WARMUP_EPOCHS
        self.no_aug_epochs = NO_AUG_EPOCHS
        self.eval_interval = EVAL_INTERVAL
        self.data_num_workers = DATA_WORKERS
        self.seed = SEED

        self.exp_name = "yolox_s_humvee_source6"
        self.output_dir = str(OUTPUT_ROOT)

    def get_dataset(self, cache=False, cache_type="ram"):
        return COCODataset(
            data_dir=self.data_dir,
            json_file=self.train_ann,
            name="",
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=50,
                flip_prob=self.flip_prob,
                hsv_prob=self.hsv_prob,
            ),
            cache=cache,
            cache_type=cache_type,
        )

    def get_eval_dataset(self, **kwargs):
        annotation_file = (
            self.test_ann if kwargs.get("testdev", False) else self.val_ann
        )
        return COCODataset(
            data_dir=self.data_dir,
            json_file=annotation_file,
            name="",
            img_size=self.test_size,
            preproc=ValTransform(legacy=kwargs.get("legacy", False)),
        )

    def get_evaluator(
        self,
        batch_size,
        is_distributed,
        testdev=False,
        legacy=False,
    ):
        return StandardCOCOEvaluator(
            dataloader=self.get_eval_loader(
                batch_size,
                is_distributed,
                testdev=testdev,
                legacy=legacy,
            ),
            img_size=self.test_size,
            confthre=self.test_conf,
            nmsthre=self.nmsthre,
            num_classes=self.num_classes,
            testdev=testdev,
        )


def checkpoint_compatibility(checkpoint_path: Path) -> dict[str, Any]:
    checkpoint_path = checkpoint_path.resolve()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)

    checkpoint_sha256 = sha256_file(checkpoint_path)
    if checkpoint_sha256 != EXPECTED_CARPARTS_CHECKPOINT_SHA256:
        raise ValueError(
            "Carparts checkpoint SHA-256 does not match the promoted "
            f"{CARPARTS_PARENT_VERSION} artifact."
        )

    payload = torch.load(checkpoint_path, map_location="cpu")
    state = payload.get("model") if isinstance(payload, dict) else None
    if not isinstance(state, dict):
        raise ValueError(
            "Expected a YOLOX checkpoint with a top-level 'model' state dict."
        )

    target_state = HumveeSource6Experiment().get_model().state_dict()
    compatible = [
        key
        for key, value in target_state.items()
        if key in state and tuple(state[key].shape) == tuple(value.shape)
    ]
    shape_mismatches = [
        key
        for key, value in target_state.items()
        if key in state and tuple(state[key].shape) != tuple(value.shape)
    ]
    missing = [key for key in target_state if key not in state]

    if not compatible:
        raise ValueError("Checkpoint has no shape-compatible YOLOX tensors.")

    return {
        "path": str(checkpoint_path),
        "version": CARPARTS_PARENT_VERSION,
        "sha256": checkpoint_sha256,
        "compatible_tensors": len(compatible),
        "shape_mismatches": len(shape_mismatches),
        "missing_tensors": len(missing),
        "sample_shape_mismatches": shape_mismatches[:10],
    }


if CARPARTS_CHECKPOINT.is_file():
    print(checkpoint_compatibility(CARPARTS_CHECKPOINT))
else:
    print(
        "Promoted checkpoint is not present locally. Restore it at "
        f"{CARPARTS_CHECKPOINT} before enabling training."
    )


### 4. Define the modular training harness

The harness refuses to overwrite an existing run directory. Use a new run name or deliberately archive the old run before repeating an experiment.


In [ ]:
@dataclass(frozen=True)
class RunSpecification:
    name: str
    initialization: str
    checkpoint: Path | None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_trainer_args(run: RunSpecification) -> argparse.Namespace:
    return argparse.Namespace(
        experiment_name=run.name,
        name=None,
        dist_backend="nccl",
        dist_url=None,
        batch_size=BATCH_SIZE,
        devices=1,
        exp_file=str(DETECTION_DIR / "humvee_initialization_comparison.ipynb"),
        resume=False,
        ckpt=str(run.checkpoint) if run.checkpoint else None,
        start_epoch=None,
        num_machines=1,
        machine_rank=0,
        fp16=FP16,
        cache=None,
        occupy=False,
        logger="tensorboard",
        opts=[],
    )


def write_run_contract(
    run: RunSpecification,
    run_directory: Path,
) -> None:
    contract = {
        "run": asdict(run),
        "seed": SEED,
        "classes": list(export_categories),
        "carparts_parent_version": CARPARTS_PARENT_VERSION,
        "expected_carparts_checkpoint_sha256": (
            EXPECTED_CARPARTS_CHECKPOINT_SHA256
        ),
        "source_annotation": str(SOURCE_ANNOTATIONS),
        "source_annotation_sha256": sha256_file(SOURCE_ANNOTATIONS),
        "split_annotations": {
            split: str(ANNOTATION_DIRECTORY / file_name)
            for split, file_name in ANNOTATION_FILES.items()
        },
        "group_column": GROUP_COLUMN,
        "split_fractions": SPLIT_FRACTIONS,
        "input_size": INPUT_SIZE,
        "test_size": TEST_SIZE,
        "max_epochs": MAX_EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "no_aug_epochs": NO_AUG_EPOCHS,
        "eval_interval": EVAL_INTERVAL,
        "batch_size": BATCH_SIZE,
        "data_workers": DATA_WORKERS,
        "fp16": FP16,
        "initialization_checkpoint_sha256": (
            sha256_file(run.checkpoint) if run.checkpoint else None
        ),
    }
    contract["run"]["checkpoint"] = (
        str(run.checkpoint) if run.checkpoint else None
    )
    (run_directory / "run_contract.json").write_text(
        json.dumps(contract, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def train_one(run: RunSpecification) -> dict[str, Any]:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required by this YOLOX training harness.")
    if run.checkpoint is not None:
        checkpoint_compatibility(run.checkpoint)

    run_directory = OUTPUT_ROOT / run.name
    if run_directory.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing run directory: {run_directory}"
        )

    run_directory.mkdir(parents=True)
    write_run_contract(run, run_directory)

    seed_everything(SEED)
    experiment = HumveeSource6Experiment()
    trainer = Trainer(experiment, make_trainer_args(run))

    try:
        trainer.train()
        best_ap = float(trainer.best_ap)
    finally:
        del trainer
        gc.collect()
        torch.cuda.empty_cache()

    best_checkpoint = run_directory / "best_ckpt.pth"
    if not best_checkpoint.is_file():
        raise FileNotFoundError(
            f"Training completed without a best checkpoint: {best_checkpoint}"
        )

    return {
        "run": run.name,
        "initialization": run.initialization,
        "validation_ap50_95": best_ap,
        "best_checkpoint": str(best_checkpoint),
        "best_checkpoint_sha256": sha256_file(best_checkpoint),
    }


### 5. Launch both controlled runs

Restore the promoted checkpoint at the configured path, review every parameter above, and then change RUN_TRAINING to True. This cell runs scratch first and carparts23-yolox-s-coco-v1 initialized second. All data, scheduling, augmentation, evaluation, seed, and hardware parameters are otherwise shared.


In [ ]:
comparison_results: list[dict[str, Any]] = []

if RUN_TRAINING:
    if not CARPARTS_CHECKPOINT.is_file():
        raise FileNotFoundError(
            "Restore the promoted Carparts checkpoint at "
            f"{CARPARTS_CHECKPOINT} before enabling the comparison."
        )
    checkpoint_compatibility(CARPARTS_CHECKPOINT)

    run_specifications = (
        RunSpecification(
            name="scratch_seed42",
            initialization="random",
            checkpoint=None,
        ),
        RunSpecification(
            name="carparts_initialized_seed42",
            initialization=CARPARTS_PARENT_VERSION,
            checkpoint=CARPARTS_CHECKPOINT.resolve(),
        ),
    )

    for run_specification in run_specifications:
        comparison_results.append(train_one(run_specification))

    result_path = OUTPUT_ROOT / "initialization_comparison.json"
    result_path.write_text(
        json.dumps(comparison_results, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )
    print(f"Wrote comparison results to {result_path}")
else:
    print(
        "Training is disabled. Restore the promoted checkpoint and set "
        "RUN_TRAINING=True after reviewing the notebook."
    )


## Checks

### Compare validation performance

This cell reads the best validation AP stored by each YOLOX run. It does not evaluate on the reserved test split.


In [ ]:
def collect_existing_results() -> list[dict[str, Any]]:
    run_definitions = (
        ("scratch_seed42", "random"),
        ("carparts_initialized_seed42", CARPARTS_PARENT_VERSION),
    )
    results = []

    for run_name, initialization in run_definitions:
        checkpoint_path = OUTPUT_ROOT / run_name / "best_ckpt.pth"
        if not checkpoint_path.is_file():
            continue
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
        results.append(
            {
                "run": run_name,
                "initialization": initialization,
                "validation_ap50_95": float(checkpoint["best_ap"]),
                "checkpoint": str(checkpoint_path),
            }
        )

    return results


existing_results = collect_existing_results()
if len(existing_results) != 2:
    print(
        "Both completed best checkpoints are required before comparison. "
        f"Found {len(existing_results)}."
    )
else:
    print("run | initialization | validation AP50:95")
    print("--- | --- | ---:")
    for result in existing_results:
        print(
            f"{result['run']} | {result['initialization']} | "
            f"{result['validation_ap50_95']:.4f}"
        )

    scratch_ap = next(
        result["validation_ap50_95"]
        for result in existing_results
        if result["initialization"] == "random"
    )
    initialized_ap = next(
        result["validation_ap50_95"]
        for result in existing_results
        if result["initialization"] == CARPARTS_PARENT_VERSION
    )
    print(
        "Carparts initialization delta: "
        f"{initialized_ap - scratch_ap:+.4f} AP50:95"
    )


## Next Steps

1. Restore carparts23-yolox-s-coco-v1 at the configured path and verify its recorded SHA-256.
2. Populate source grouping and license metadata, then rerun the same comparison with GROUP_COLUMN set for leakage-resistant splits.
3. Compare validation AP50:95 and per-class behavior; do not select using the test split.
4. After selecting the initialization strategy, evaluate that one checkpoint once on the reserved test annotations.
5. Keep the source-export taxonomy attached to this experiment. Do not report it as the canonical CRATER six-class detector.
